In [1]:
%pip install transformers datasets evaluate -q

%pip install accelerate -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import AutoTokenizer,TrainingArguments, AutoModelForTokenClassification
from datasets import Dataset
import evaluate
import os

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"✅ Device set to: {device}")

/Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Device set to: mps


In [ ]:
def load_conll(filepath):
    with open(filepath, encoding="utf-8") as f:
        sentences = []
        labels = []
        current_tokens, current_tags = [], []

        for line in f:
            line = line.strip()
            if not line:
                if current_tokens:
                    sentences.append(current_tokens)
                    labels.append(current_tags)
                    current_tokens, current_tags = [], []
                continue

            splits = line.split()
            if len(splits) >= 2:
                token = splits[0]
                tag = splits[-1]
                current_tokens.append(token)
                current_tags.append(tag)

        if current_tokens:
            sentences.append(current_tokens)
            labels.append(current_tags)

    return Dataset.from_dict({"tokens": sentences, "ner_tags": labels})

dataset_path = "../data/processed/ethiopic_news_ner.conll"
telegram_dataset = load_conll(dataset_path)
telegram_dataset = telegram_dataset.train_test_split(test_size=0.1)
telegram_dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 89
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 10
    })
})

In [4]:
label_list = ['B-Product', 'I-Product', 'B-LOC', 'I-LOC', 'B-PRICE', 'I-PRICE', 'O']
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
print(label2id)

{'B-Product': 0, 'I-Product': 1, 'B-LOC': 2, 'I-LOC': 3, 'B-PRICE': 4, 'I-PRICE': 5, 'O': 6}


In [5]:
model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
assert tokenizer.is_fast

In [6]:
unique_tags = set(tag for sequence in telegram_dataset["train"]["ner_tags"] for tag in sequence)
label_list = sorted(list(unique_tags))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print("📌 Label list:", label_list)

def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    word_ids = tokenized_inputs.word_ids()
    label_ids = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            tag = example["ner_tags"][word_idx]
            label_ids.append(label2id.get(tag, label2id["O"]))
        else:
            prev_tag = example["ner_tags"][word_idx]
            new_tag = prev_tag.replace("B-", "I-") if prev_tag.startswith("B-") else prev_tag
            label_ids.append(label2id.get(new_tag, label2id["O"]))
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs

tokenized_dataset = telegram_dataset.map(tokenize_and_align_labels)

📌 Label list: ['B-LOC', 'B-PRICE', 'B-PRODUCT', 'I-LOC', 'I-PRICE', 'I-PRODUCT', 'O']


Map: 100%|██████████| 10/10 [00:00<00:00, 2518.80 examples/s]


In [7]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
).to(device)

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)

    true_labels = [
        [id2label[l] for l in label if l != -100]
        for label in labels
    ]
    true_preds = [
        [id2label[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ner_results",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
)

In [10]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/var/folders/9c/z436vv391wd5csdhpxszxk600000gn/T/ipykernel_21921/3458185262.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
trainer.train()

/Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


TrainOutput(global_step=60, training_loss=0.6115058898925781, metrics={'train_runtime': 26.4002, 'train_samples_per_second': 16.856, 'train_steps_per_second': 2.273, 'total_flos': 29070578254080.0, 'train_loss': 0.6115058898925781, 'epoch': 5.0})

In [12]:
model_dir = "amharic-ner-model"
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

('amharic-ner-model/tokenizer_config.json',
 'amharic-ner-model/special_tokens_map.json',
 'amharic-ner-model/tokenizer.json')

In [13]:
from transformers import pipeline

model_path = "amharic-ner-model"

ner = pipeline("ner", model=model_path, tokenizer=model_path, aggregation_strategy="simple", device=0 if torch.backends.mps.is_available() else -1)

Device set to use mps:0


In [14]:
test_text = "በአዲስ አበባ በ 1000 ብር የሚሸጥ የሴቶች ሻሚዝ አለ።"

entities = ner(test_text)

if not entities:
    print("⚠️ No entities found.")
else:
    for ent in entities:
        print(f"🟢 {ent['word']} → {ent['entity_group']} ({ent['score']:.2f})")

⚠️ No entities found.


In [16]:
test_samples = [
    "በአዲስ አበባ በ 1000 ብር የሚሸጥ የሴቶች ሻሚዝ",
    "ከአዲስ አበባ እስከ ሀዋሳ በ 150 ብር የድልድይ አገልግሎት አለ",
    "የልጆች ጫማ በ 500 ብር በቦሌ",
    "ነፃ ማድረሻ አለ። በስልክ ቁጥር 0912345678 ይደውሉ"
]

for text in test_samples:
    print(f"\n📨 {text}")
    results = ner(text)
    if not results:
        print("⚠️ No entities found.")
    else:
        for ent in results:
            print(f"🟢 {ent['word']} → {ent['entity_group']} ({ent['score']:.2f})")



📨 በአዲስ አበባ በ 1000 ብር የሚሸጥ የሴቶች ሻሚዝ
⚠️ No entities found.

📨 ከአዲስ አበባ እስከ ሀዋሳ በ 150 ብር የድልድይ አገልግሎት አለ
⚠️ No entities found.

📨 የልጆች ጫማ በ 500 ብር በቦሌ
⚠️ No entities found.

📨 ነፃ ማድረሻ አለ። በስልክ ቁጥር 0912345678 ይደውሉ
⚠️ No entities found.
